# SampleTrace walkthrough — no Benchling tenant required

This notebook uses **mock mode** end-to-end: synthetic Benchling responses bundled with the package, plus the sample sheet fixture in the test suite. You can run every cell without ever touching a real Benchling tenant.

When you're ready to wire up real Benchling, swap `BenchlingClient.mock()` for `BenchlingClient.from_yaml('your_config.yml')` and set `BENCHLING_API_KEY` in your environment. Everything downstream stays the same.

In [ ]:
from pathlib import Path

from sampletrace.benchling_client import BenchlingClient
from sampletrace.parsers import parse_sample_sheet, parse_count_matrix
from sampletrace.reconciler import ReconcilerConfig, reconcile

## 1. Pull sample registrations from (mock) Benchling

In [ ]:
client = BenchlingClient.mock()
benchling_samples = client.fetch_samples()
print(f'{len(benchling_samples)} samples from Benchling:')
for s in benchling_samples:
    print(f'  {s.sample_id:12}  organism={s.organism!r}  i7={s.index_i7}')

## 2. Parse the sample sheet (v2 BCL Convert format)

In [ ]:
fixtures = Path('../tests/fixtures')
sheet_samples = parse_sample_sheet(fixtures / 'SampleSheet_v2.csv')
print(f'{len(sheet_samples)} samples from sample sheet:')
for s in sheet_samples:
    print(f'  {s.sample_id:12}  lane={s.lane}  i7={s.index_i7}')

## 3. Parse a count matrix (just column headers)

In [ ]:
matrix_samples = parse_count_matrix(fixtures / 'counts.tsv')
print(f'{len(matrix_samples)} samples in count matrix:')
for s in matrix_samples:
    print(f'  {s.sample_id}')

## 4. Reconcile the three sources

In [ ]:
downstream = sheet_samples + matrix_samples
report = reconcile(
    benchling_samples,
    downstream,
    config=ReconcilerConfig(fuzzy_threshold=80, high_confidence=95),
    run_id='demo_run',
)
print('summary:', report.summary())
for row in report.rows:
    mark = '✓' if not row.is_flagged else '!'
    print(f'  {mark} {row.sample_id:12} {row.confidence.value:8} {row.mismatches}')

## 5. Try a drift scenario

Now let's swap in a Benchling fixture where the metadata has drifted (one sample changed organism, another renamed).

In [ ]:
drift_client = BenchlingClient.mock(fixtures / 'benchling_with_drift.json')
drift_bch = drift_client.fetch_samples()
drift_report = reconcile(drift_bch, sheet_samples)
print('summary:', drift_report.summary())
for row in drift_report.flagged:
    print(f'  ! {row.sample_id:12} {row.confidence.value:8}')
    for note in row.notes:
        print(f'       {note}')